# Two-locus Island model

In [3]:
import phasic as ph
import optax
import numpy as np
import jax.numpy as jnp
import pandas as pd
from itertools import combinations_with_replacement as combs
from typing import Optional
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
%config InlineBackend.figure_format = 'svg'
from tqdm.auto import tqdm
from vscodenb import set_vscode_theme

np.random.seed(42)
set_vscode_theme()
sns.set_palette('tab10')

# set_log_level('DEBUG') 

#from phasic import GraphCache                                                              
                                                                                                         
graph_cache = ph.GraphCache() 

## Two population ghost model

In [7]:

def two_loc_island(state, indexer=None):

    pop1_coal_idx = 0
    pop2_coal_idx = 1
    pop1_mig_idx =  2
    pop2_mig_idx =  3
    rec_idx =  4

    coef_len = 5

    transitions = []

    nonzero_idx = np.nonzero(state[:indexer.descendants.state_length])[0]

    if state.sum() <= 1: return transitions
    
    for i, j in combs(nonzero_idx, 2):

        if state[i] == 0: continue
        props_i = indexer.descendants.index_to_props(i)
        if state[j] == 0: continue
        props_j = indexer.descendants.index_to_props(j)
        if props_j.in_pop != props_i.in_pop:
            # skip if in different pops            
            continue
        same = int(i == j)
        if same and state[i] < 2:
            # skip if same and less than two lineages
            continue
        if not same and (state[i] < 1 or state[j] < 1):
            # skip if different and either kind has no lineages
            continue 

        # coalescence
        loc1=props_i.loc1 + props_j.loc1
        loc2=props_i.loc2 + props_j.loc2
        if loc1 <= nr_samples and loc2 <= nr_samples:
            child = state.copy()
            child[i] -= 1
            child[j] -= 1
            k = indexer.descendants.props_to_index(
                loc1=props_i.loc1 + props_j.loc1,
                loc2=props_i.loc2 + props_j.loc2,
                in_pop=props_i.in_pop
                )
            child[k] += 1

            coef = np.zeros(coef_len)
            if props_i.in_pop == 1:
                coef[pop1_coal_idx] = state[i]*(state[j]-same)/(1+same)
            else:
                coef[pop2_coal_idx] = state[i]*(state[j]-same)/(1+same)
            transitions.append([child, coef])

    for i in nonzero_idx:
        props_i = indexer.descendants.index_to_props(i)
        
        # recombination
        loc1 = props_i.loc1 
        loc2 = props_i.loc2
        if state[i] > 0 and 0 < loc1 <= nr_samples and 0 < loc2 <= nr_samples:
            child = state.copy()
            child[i] -= 1
            k = indexer.descendants.props_to_index(
                loc1=0,
                loc2=props_i.loc2, 
                in_pop=props_i.in_pop
                )
            child[k] += 1
            k = indexer.descendants.props_to_index(
                loc1=props_i.loc1,
                loc2=0, 
                in_pop=props_i.in_pop
                )
            child[k] += 1
            coef = np.zeros(coef_len)
            if props_i.in_pop == 1:
                coef[rec_idx] = state[i]
            else:
                coef[rec_idx] = state[i]
            transitions.append([child, coef])            

        # migration
        if state[i] > 0:
            child = state.copy()
            other_pop = 2 if props_i.in_pop == 1 else 1
            child = state.copy()
            child[i] -= 1
            k = indexer.descendants.props_to_index(
                loc1=props_i.loc1,
                loc2=props_i.loc2, 
                in_pop=other_pop
                )
            child[k] += 1
            coef = np.zeros(coef_len)
            if props_i.in_pop == 2:
                 coef[pop1_mig_idx] = state[i]
            else:
                coef[pop2_mig_idx] = state[i]
            transitions.append([child, coef])                      

    return transitions


nr_samples = 4

coal_rate_pop1, coal_rate_pop2 = 1, 1
mig_rate_pop1, mig_rate_pop2 = 1, 1
rec_rate = 5

true_theta = [coal_rate_pop1, coal_rate_pop2, mig_rate_pop1, mig_rate_pop2, rec_rate]

coef_len = len(true_theta)

indexer = ph.StateIndexer(
    descendants=[        
        ph.Property('loc1', min_value=0, max_value=nr_samples),
        ph.Property('loc2', min_value=0, max_value=nr_samples),
        ph.Property('in_pop', min_value=1, max_value=2),
    ],
    slots=['epoch'] 
)
indexer.state_length
initial = [0] * indexer.state_length

# set initial state with all lineages having one descendant at both loci
initial[indexer.descendants.props_to_index(loc1=1, loc2=1, in_pop=1, 
                                           )] = nr_samples

graph = ph.Graph(two_loc_island, ipv=initial, indexer=indexer)   

# graph_cache.save_graph(graph, two_locus_ghost_island, epoch_idx=0, nr_epochs=2, nr_params=6, indexer=indexer)  

graph.update_weights(true_theta)
print(graph.vertices_length())
graph.plot(rankdir='TB',# size=(12,8),
           max_nodes=200,
           label_fmt=False,
            )

1432
Graph has too many nodes (1432). Please set max_nodes to a higher value.


In [8]:
mutation_rate = 1 #.2e-4
reward_limit=1

joint_prob_graph = graph.joint_prob_graph(indexer,
                               reward_only=['loc1', 'loc2'],
                               reward_limit=reward_limit, 
                               mutation_rate=mutation_rate
                               )

In [9]:
joint_prob_graph.plot(rankdir='TB', size=(10,10),
           nodesep=0.1,
           ranksep=5,
           max_nodes=700,
           wrap=10,
           label_fmt=lambda state: None
            )

Graph has too many nodes (140436). Please set max_nodes to a higher value.


In [10]:
joint_prob_graph.update_weights(true_theta + [mutation_rate]) 

In [ ]:
joint_prob_table = joint_prob_graph.joint_prob_table()
print(f'Deficit: {1-joint_prob_table.prob.sum():.2g}')
joint_prob_table

In [ ]:
loc1_feat = [c for c in joint_prob_table.columns if c.startswith('loc1')]
loc2_feat = [c for c in joint_prob_table.columns if c.startswith('loc2')]
features = loc1_feat + loc2_feat

In [ ]:
# Remove rows without one ton at each locus
# obs_joint_prob_table = joint_prob_table.loc[
#     (joint_prob_table[loc1_feat].sum(axis=1) > 0) & (joint_prob_table[loc2_feat].sum(axis=1) > 0)
#     ]

In [ ]:
def sample_joint_observations(joint_prob_graph, theta, nr_observations):
    joint_prob_graph.update_weights(theta) 
    joint_prob_table = joint_prob_graph.joint_prob_table()
    p = joint_prob_table['prob'] / joint_prob_table['prob'].sum()
    p = p.to_numpy()
    sample = np.random.choice(joint_prob_table.index.values, nr_observations, p=p)
    observations = joint_prob_table.loc[sample, joint_prob_table.columns[:-1]].to_numpy().tolist()
    return observations


observations = pd.DataFrame.from_records(
    sample_joint_observations(joint_prob_graph, [*true_theta, mutation_rate], nr_observations=10000),
    columns=features
    )
print(observations.index.size)

# Filter out observations represented by deficit:
obs_idx = observations.set_index(features)
obs_jpt_idx = joint_prob_table.set_index(features)
subset = observations[obs_idx.index.isin(obs_jpt_idx.index)]
print(subset.index.size)

observations = subset.to_numpy()

In [ ]:
%%monitor

joint_prob_graph_cont = graph.joint_prob_graph(indexer,
                               reward_only=['loc1', 'loc2'],
                               reward_limit=reward_limit, 
                               mutation_rate=mutation_rate,
                                          discrete=False)
svgd = joint_prob_graph_cont.svgd(
    observations, 
    fixed=[(4, rec_rate), (5, mutation_rate)],
    # prior=LogGaussPrior(ci=[1/50_000, 1/5000]),
    # prior=GaussPrior(ci=[0.5, 5]),
    n_iterations=200,
    n_particles=20,
    optimizer=optax.adamw(learning_rate=0.02)
    # optimizer=Adam(learning_rate=0.2),
    # learning_rate=learning_rate,
    # epoch_starts=[0, 0.01]
    # tied=[(0, [0, 1])], # tie coal rate across epochs
    # daisy_chain_t_eval=30,
    # daisy_chain_t_eval='auto',
    # daisy_chain_t_eval_tol=1e-6,           # default; tighter = more conservative
    # daisy_chain_probe_theta=[5.0, 1.0],    # optional; defaults to ones
)

In [ ]:
svgd.plot_ci(true_theta=true_theta) ;

In [ ]:
svgd.plot_convergence()

In [ ]:
svgd.plot_trace(['coal pop1', 'coal pop2', 'mig pop1', 'mig pop2', 'rec rate'], true_theta=true_theta)

In [ ]:
svgd.plot_pairwise(true_theta=true_theta, )